In [1]:
import pandas as pd
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from torchvision import models
import torch.nn as nn
from torch.utils.data import ConcatDataset
import torch.optim as optim

dataset : https://www.kaggle.com/datasets/puneet6060/intel-image-classification?select=seg_train

Create the label files

In [21]:
dataset_train_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_train/seg_train'
dataset_test_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_test/seg_test'

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}

for c in classe.keys():
    label_train = []
    label_test = []
    
    class_train_path = os.path.join(dataset_train_path, c)
    class_test_path = os.path.join(dataset_test_path, c)
    
    img_train_path = class_train_path +'/img'
    img_test_path = class_test_path+'/img'

    for imag in os.listdir(img_train_path):
        if not imag.startswith('.'): # avoid .DS_Store
            label_train.append({'image_name' : imag, 'label' : classe[c]})
        

    for imag in os.listdir(img_test_path):
        if not imag.startswith('.'):
            label_test.append({'image_name' : imag, 'label' : classe[c]})
        
        
    df_train = pd.DataFrame(label_train)
    df_train.to_csv(os.path.join(class_train_path, f"{classe[c]}_label_train.csv"), index=False)
    df_test = pd.DataFrame(label_test)
    df_test.to_csv(os.path.join(class_test_path, f"{classe[c]}_label_test.csv"), index=False)



Repartition of the dataset through the 6 classes 

create our dataset 

In [22]:
#For each task we will do a classification on 2 classes, so we will create a new dataset with only 2 classes by merging our 2 directories and same thing for the file with the labels. 

def task(A,B, idx):
    '''A and B are the 2 classes of the idx task'''
    path_img_A_train = os.path.join(dataset_train_path, A, 'img')
    path_img_B_train = os.path.join(dataset_train_path, B, 'img')
    path_img_AB_train = os.path.join(dataset_train_path, f'df_{A}_{B}', 'img')
    path_img_A_test = os.path.join(dataset_test_path, A, 'img')
    path_img_B_test = os.path.join(dataset_test_path, B, 'img')
    path_img_AB_test = os.path.join(dataset_test_path, f'df_{A}_{B}', 'img')

    #create the directory if it does not exist
    os.makedirs(path_img_AB_train, exist_ok=True)
    os.makedirs(path_img_AB_test, exist_ok=True)

    def move(init, final):
        for file in os.listdir(init):
            if not file.startswith('.'):
                source_file = os.path.join(init, file)
                dest_file = os.path.join(final, file)
                os.rename(source_file, dest_file)



    move(path_img_A_train, path_img_AB_train)
    move(path_img_B_train, path_img_AB_train)

    move(path_img_A_test, path_img_AB_test)
    move(path_img_B_test, path_img_AB_test)

    label_A_train = pd.read_csv(os.path.join(dataset_train_path, A, f"{classe[A]}_label_train.csv"))
    label_B_train = pd.read_csv(os.path.join(dataset_train_path, B, f"{classe[B]}_label_train.csv"))
    label_A_train['label'] = 0
    label_B_train['label'] = 1
    label_AB_train = pd.concat([label_A_train, label_B_train], ignore_index=True)
    label_AB_train['task_id'] = idx
    label_AB_train.to_csv(os.path.join(dataset_train_path,f'df_{A}_{B}', 'label_train.csv'), index=False)



    label_A_test = pd.read_csv(os.path.join(dataset_test_path, A, f"{classe[A]}_label_test.csv"))
    label_B_test = pd.read_csv(os.path.join(dataset_test_path, B, f"{classe[B]}_label_test.csv"))
    label_A_test['label'] = 0
    label_B_test['label'] = 1
    label_AB_test = pd.concat([label_A_test, label_B_test], ignore_index=True)
    label_AB_test['task_id'] = idx
    label_AB_test.to_csv(os.path.join(dataset_test_path, f'df_{A}_{B}', 'label_test.csv'), index=False)
    
    return (path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test)
    
path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test = task('buildings', 'sea', 0)
path_img_CD_train, path_img_CD_test, label_CD_train, label_CD_test = task('forest', 'mountain', 1)

In [ ]:
"""

def portion_dataset(images,label, portion): #returns a portion of the dataset (images and labels) given a portion value between 0 and 1. 
    
    nb = int(len(label) * portion) #number of images in the dataset that we will keep in order to avoid catastrophic forgetting during the new task.

    idx = np.random.choice(len(label), size=nb, replace=False) #indexes of the images that we will keep.

    img_portion = images[idx]
    label_portion = label[idx]
    
    return img_portion, label_portion
    
"""

In [23]:
class CustomImageDataset(Dataset):

    def __init__(self, annotations_file, img_dir, portion, img_transform=None): # portion is the pourcentage of the dataset that we will keep in order to avoid catastrophic forgetting during the new task.

        self.img_dir = img_dir
        self.img_transform = img_transform
        
        if portion < 1.0:
            self.img_labels = annotations_file.sample(frac=portion, random_state=42).reset_index(drop=True)
        else:
            self.img_labels = annotations_file

    def __len__(self): #Return the number of samples in the dataset.
        return len(self.img_labels)
   
    def __getitem__(self, idx): #For a given index, it returns the sample (image, label) associated to it.
        
        # Get the file path of the image by combining the directory and the image filename from the labels DataFrame
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        
        # Read the image from the specified file path and convert it to a tensor
        image = Image.open(img_path)
        
        # Retrieve the corresponding label for the image from the labels DataFrame
        label = self.img_labels.iloc[idx, 1]
        
        task = self.img_labels.iloc[idx, 2]
        
        # If a transform function is provided, apply it to the image
        if self.img_transform:
            image = self.img_transform(image)
            # If a target_transform function is provided, apply it to the label
        
        # Return the transformed image and label as a tuple
        return image, label, task
            

In [5]:
# List of transformations that will be applied to images
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])


In [24]:

dataset_train_AB_full = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = 1 , img_transform=transform)
dataset_test_AB_full = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = 1 , img_transform=transform)

train_dataloader_AB_full = DataLoader(dataset_train_AB_full, batch_size=64, shuffle=True)
test_dataloader_AB_full= DataLoader(dataset_test_AB_full, batch_size=64, shuffle=True)


portion = 0.5 #portion of the dataset of the previous dataset we want to keep for avoiding the catastrophic forgetting during the new task.



dataset_train_AB_part = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = portion , img_transform=transform)
dataset_test_AB_part = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = portion, img_transform=transform)

train_dataloader_AB_part = DataLoader(dataset_train_AB_part, batch_size=64, shuffle=True)
test_dataloader_AB_part= DataLoader(dataset_test_AB_part, batch_size=64, shuffle=True)





dataset_train_CD_full = CustomImageDataset(annotations_file= label_CD_train, img_dir = path_img_CD_train, portion = 1 , img_transform=transform)
dataset_test_CD_full = CustomImageDataset(annotations_file= label_CD_test, img_dir = path_img_CD_test, portion = 1, img_transform=transform)

train_dataloader_CD_full = DataLoader(dataset_train_CD_full, batch_size=64, shuffle=True)
test_dataloader_CD_full= DataLoader(dataset_test_CD_full, batch_size=64, shuffle=True)






dataset_train_ABCD = ConcatDataset([dataset_train_AB_part, dataset_train_CD_full])
dataset_test_ABCD = ConcatDataset([dataset_test_AB_part, dataset_test_CD_full])

train_dataloader_ABCD = DataLoader(dataset_train_ABCD, batch_size=64, shuffle=True)
test_dataloader_ABCD= DataLoader(dataset_test_ABCD, batch_size=64, shuffle=True)


In [ ]:

"""
def reduce_dataset(dataset, portion):
    # portion = the portion of the dataset of the previous tasks we want to keep for avoiding the catastrophic forgetting during the new task. 
    nb = int(len(dataset) * portion) #number of images to keep

    # 2. Générer des indices aléatoires uniques
    indices = np.random.choice(len(dataset), size=nb, replace=False)
    return Subset(dataset, indices)

# 3. Créer le sous-jeu de données et le passer au DataLoader standard
dataset_reduit = Subset(dataset_train, indices)
train_loader = DataLoader(dataset_reduit, batch_size=64, shuffle=True)

"""

In [ ]:
basic_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = basic_model.fc.in_features
basic_model.fc = nn.Linear(num_features, 2)

In [ ]:
def train_loop_basic(dataloader, model, loss, optimizer, num_epochs, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.train()
    
    for epoch in range(num_epochs):
        
        epoch_loss = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0}  # accuracy dictionary for each class
        
        for image, label, _ in dataloader:

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)    
            
            optimizer.zero_grad()  # Set gradient to zero
            
            # Compute prediction and loss
            pred = model(image)
                
            batch_loss = loss(pred, label)
            epoch_loss += batch_loss.item() 
                
            pred_class = pred.argmax(dim=1)
            pred_classA = (pred_class == 0)
            label_A = (label == 0)
            dic_acc["A_true"] += (pred_classA & label_A).sum().item()
            dic_acc["A_tot"] += label_A.sum().item()

            pred_classB = (pred_class == 1)
            label_B = (label == 1)
            dic_acc["B_true"] += (pred_classB & label_B).sum().item()
            dic_acc["B_tot"] += label_B.sum().item()

            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        epoch_loss = epoch_loss / size
        
            
        print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss:.4f}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0):.2f} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0):.2f} %, '
    )


    

In [25]:
class rehearsal(nn.Module):
# Constructor method to initialize layers of the network def __init__(self):
        def __init__(self,num_classe=2):
                super().__init__()
                self.num_classe = num_classe
                self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # we consider a pretrained model 
                self.num_features = self.model.fc.in_features
                self.model.fc = nn.Identity()
                self.heads = nn.ModuleDict()
        
        def add_task(self, num_task):
                num_task = str(num_task)  
                if num_task not in self.heads:
                        self.heads[num_task] = nn.Linear(self.num_features, self.num_classe)
                
        # Forward pass of the network
        def forward(self, x, num_task):
                x = self.model(x)
                num_task = num_task[0].item() if isinstance(num_task, torch.Tensor) else num_task
                num_task = str(num_task)  
                if num_task in self.heads:
                        return self.heads[num_task](x)
                else:
                        raise ValueError(f"Task {num_task} not found in the model.")


In [33]:
def train_loop_rehearsal(dataloader, model, loss, optimizer, num_epochs, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.train()
    
    for epoch in range(num_epochs):
        
        epoch_lossAB = 0.0
        epoch_lossCD = 0.0
        epoch_loss_tot = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class
        
        for image, label, task in dataloader:
            
            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0
            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)    
            
            optimizer.zero_grad()  # Set gradient to zero
            
            # Compute prediction and loss
            pred = model(image, task)
            
            mask_AB = (task == 0)
            mask_CD = (task == 1)
            
            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]   
                label_AB = label[mask_AB] 
                
                batch_lossAB = loss(pred_AB, label_AB)
                epoch_lossAB += batch_lossAB.item() 
                
                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()
                    
            
            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]   
                label_CD = label[mask_CD] 
                
                batch_lossCD = loss(pred_CD, label_CD)
                epoch_lossCD += batch_lossCD.item()
                
                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()
 
            
            batch_loss_tot = batch_lossAB + batch_lossCD
            epoch_loss_tot += batch_loss_tot.item()

            # Backpropagation
            batch_loss_tot.backward()
            optimizer.step()
            
        epoch_lossAB = epoch_lossAB / size
        epoch_lossCD = epoch_lossCD / size
        epoch_loss = epoch_loss_tot / size
        
            
    print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss:.4f}, 1st task Loss: {epoch_lossAB:.4f}, 2nd task Loss: {epoch_lossCD:.4f}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0):.2f} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0):.2f} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0):.2f} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0):.2f} %')


    

In [27]:
def test_loop_rehearsal(dataloader, model, loss, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.eval()
    loss_test = 0.0
    loss_testAB = 0.0
    loss_testCD = 0.0
    
    dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class
   
    with torch.no_grad():
        for image, label, task in dataloader:
            
            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0
            
            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)    
            
            # Compute prediction and loss
            pred = model(image, task)
            
            mask_AB = (task == 0)
            mask_CD = (task == 1)
            
            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]   
                label_AB = label[mask_AB] 
                
                batch_lossAB = loss(pred_AB, label_AB)
                loss_testAB += batch_lossAB.item()
                
                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()
                
                    
            
            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]   
                label_CD = label[mask_CD] 
                
                batch_lossCD = loss(pred_CD, label_CD)
                loss_testCD += batch_lossCD.item()
                
                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()

        
        loss_testAB = loss_testAB / size
        loss_testCD = loss_testCD / size   
        loss_test =  loss_testAB + loss_testCD

        
            
        print(f'Total Mean Loss per images: {loss_test}, 1st task Loss: {loss_testAB}, 2nd task Loss: {loss_testCD}, Accuracy A : {dic_acc["A_true"] *100 / dic_acc["A_tot"]} %, B :{dic_acc["B_true"] *100 / dic_acc["B_tot"]} %, C :{dic_acc["C_true"] *100 / dic_acc["C_tot"]} %, D :{dic_acc["D_true"] *100 / dic_acc["D_tot"]} %)')



In [34]:
model_rehearsal = rehearsal()
model_rehearsal.add_task(0)  
model_rehearsal.add_task(1)

In [29]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_rehearsal.parameters(), lr=0.001, momentum=0.9)
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
train_loop_rehearsal(train_dataloader_AB_full , model_rehearsal, loss, optimizer, num_epochs, device)

In [ ]:
test_loop_rehearsal(test_dataloader_AB_full, model_rehearsal, loss, device)

In [ ]:
train_loop_rehearsal(train_dataloader_ABCD , model_rehearsal, loss, optimizer, num_epochs, device)

In [ ]:
test_loop_rehearsal(test_dataloader_ABCD, model_rehearsal, loss, device)